## Bronze層 データ取り込み処理
### 概要
データソースから生データを取り込み、メタデータ(取り込み日時、ファイルパス)を付加した上でDelta Lake形式のテーブルへ保存する処理。

### 処理フロー
```
[データソース (CSV)]
        ↓
[Spark Stream 変換 (メタデータ付与)]
        ↓(Checkpointによる状態管理)
[Bronze Delta Table (raw_diamonds)]
```

In [0]:
from pyspark.sql.functions import current_timestamp, col

# カタログとスキーマの設定
spark.sql("USE CATALOG my_ecommerce_pipeline")
spark.sql("USE SCHEMA bronze")

# 入力データパスとチェックポイントの設定
source_dir = "/databricks-datasets/Rdatasets/data-001/csv/ggplot2/"
base_metadata_dir = "/Volumes/my_ecommerce_pipeline/bronze/checkpoints/bronze_diamonds"

schema_dir = f"{base_metadata_dir}/schema"             # スキーマ情報保持用dir
checkpoint_dir = f"{base_metadata_dir}/checkpoint/"    # 進捗状況(チェックポイント)保持用dir

# Auto Loader (cloudFiles)を用いた増分取り込みストリームの設定(定義)
bronze_stream = (
    spark.readStream
    .format("cloudFiles") # Auto Loader設定
    .option("cloudFiles.format", "csv")
    # ファイルから推論したカラム名や型定義の情報を保持するため以下のパスを指定
    .option("cloudFiles.schemaLocation", schema_dir)
    .option("header", "true")
    .load(source_dir)
    # データリネージ(系統追跡)、障害時の原因究明を容易にするために以下を設定
    .withColumn("_ingested_at", current_timestamp())           # 取り込み日時の付与
    .withColumn("_source_file", col("_metadata.file_path"))    # ソースファイル名の保持
)

# 上記で設定した定義に基づいて、Deltaテーブルへの書き込みを実行(Trigger AvailableNowで未処理分のみバッチ実行して終了)
query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    # 「ファイルをどこまで読み込んだか」という進捗状況を保持するため以下のパスを指定
    .option("checkpointLocation", checkpoint_dir)
    # 起動時点で存在する未処理データのみを処理し、完了後に自動でストリームを停止するために以下を設定
    .trigger(availableNow=True)
    .toTable("raw_diamonds")
)
 
# 書き込みジョブが完全に終了するまで待機
query.awaitTermination()

print("Bronze層への取り込みが完了しました!")